# 2. Fine-Tune a Sentence Transformer

This notebook fine-tunes a [sentence-transformers](https://www.sbert.net/) model on text-pair data
using **MultipleNegativesRankingLoss** (contrastive learning). Works for:
- Code search (query ↔ code)
- Semantic search (query ↔ document)
- Sentence similarity (sentence A ↔ sentence B)

## What you need
- A **JSON file** with `text_a` and `text_b` pairs (from notebook 01, or your own data)
- A **GPU runtime** (Colab T4 or better)

## How to customize
1. Set `DATA_PATH` to your JSON file
2. Choose a `BASE_MODEL` from [sentence-transformers models](https://huggingface.co/sentence-transformers)
3. Adjust hyperparameters
4. Run all cells

## 1. Install Dependencies

In [ ]:
!pip install -q sentence-transformers datasets

## 2. Setup

In [ ]:
import gc
import json
import os
import time
from datetime import datetime
from pathlib import Path

import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from torch.utils.data import DataLoader

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

if IN_COLAB:
    drive.mount("/content/drive")

## 3. Configuration

In [ ]:
# === Data ===
DATA_PATH = "/content/drive/MyDrive/sentence-transformer-finetune/filtered_pairs.json"

# === Model ===
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MAX_SEQ_LENGTH = 512

# === Training ===
BATCH_SIZE = 192       # large batch = more in-batch negatives (reduce if OOM)
EPOCHS = 3
LEARNING_RATE = 2.7e-5
WARMUP_RATIO = 0.05
EVAL_STEPS = 2000      # evaluate every N steps
EVAL_SAMPLES = 2000    # held-out samples for evaluation

# === Output ===
OUTPUT_DIR = "/content/drive/MyDrive/sentence-transformer-finetune/trained_model"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Data:       {DATA_PATH}")
print(f"Model:      {BASE_MODEL}")
print(f"Max tokens: {MAX_SEQ_LENGTH}")
print(f"Batch size: {BATCH_SIZE} ({BATCH_SIZE - 1} in-batch negatives)")
print(f"Epochs:     {EPOCHS}")
print(f"Output:     {OUTPUT_DIR}")

## 4. Load & Split Data

In [ ]:
print(f"Loading data from {DATA_PATH}...")
with open(DATA_PATH, "r") as f:
    all_pairs = json.load(f)

print(f"Loaded {len(all_pairs):,} pairs")

# Split into train and eval
train_pairs = all_pairs[: len(all_pairs) - EVAL_SAMPLES]
eval_pairs = all_pairs[len(all_pairs) - EVAL_SAMPLES :]

print(f"Train: {len(train_pairs):,} pairs")
print(f"Eval:  {len(eval_pairs):,} pairs")

# Create training examples
train_examples = [
    InputExample(texts=[p["text_a"], p["text_b"]])
    for p in train_pairs
]

# Create eval data for InformationRetrievalEvaluator
eval_queries = {}
eval_corpus = {}
eval_relevant_docs = {}

for i, pair in enumerate(eval_pairs):
    qid = f"q{i}"
    did = f"d{i}"
    eval_queries[qid] = pair["text_a"]
    eval_corpus[did] = pair["text_b"]
    eval_relevant_docs[qid] = {did}

print(f"Created {len(train_examples):,} training examples")
print(f"Created {len(eval_queries):,} evaluation queries")

del all_pairs, train_pairs, eval_pairs
gc.collect()

## 5. Load Model & Prepare Training

In [ ]:
print(f"\nLoading base model: {BASE_MODEL}")
model = SentenceTransformer(
    BASE_MODEL,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
model.max_seq_length = MAX_SEQ_LENGTH
print(f"Model loaded (max_seq_length={MAX_SEQ_LENGTH})")

# DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

# Loss: MultipleNegativesRankingLoss (contrastive with in-batch negatives)
train_loss = losses.MultipleNegativesRankingLoss(model)

# Evaluator
evaluator = InformationRetrievalEvaluator(
    queries=eval_queries,
    corpus=eval_corpus,
    relevant_docs=eval_relevant_docs,
    name="eval",
    show_progress_bar=True,
)

warmup_steps = int(len(train_examples) * WARMUP_RATIO / BATCH_SIZE)
total_steps = len(train_dataloader) * EPOCHS

print(f"\nTotal steps:  {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")
print(f"Eval every:   {EVAL_STEPS} steps")

## 6. Train

In [ ]:
print(f"\n{'='*60}")
print("STARTING TRAINING")
print(f"{'='*60}")

start_time = time.time()

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    evaluation_steps=EVAL_STEPS,
    output_path=OUTPUT_DIR,
    save_best_model=True,
    show_progress_bar=True,
    optimizer_params={"lr": LEARNING_RATE},
    checkpoint_save_steps=EVAL_STEPS,
    checkpoint_path=os.path.join(OUTPUT_DIR, "checkpoints"),
    use_amp=True,
)

training_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"TRAINING COMPLETE ({training_time / 3600:.2f} hours)")
print(f"Model saved to: {OUTPUT_DIR}")
print(f"{'='*60}")

## 7. Save Training Metadata

In [ ]:
# Reload and set max_seq_length in saved config
final_model = SentenceTransformer(OUTPUT_DIR)
final_model.max_seq_length = MAX_SEQ_LENGTH
final_model.save(OUTPUT_DIR)
del final_model

metadata = {
    "base_model": BASE_MODEL,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_samples": len(train_examples),
    "eval_samples": len(eval_queries),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "training_time_hours": training_time / 3600,
    "completed_at": datetime.now().isoformat(),
}

with open(os.path.join(OUTPUT_DIR, "training_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("Training metadata saved")
print(json.dumps(metadata, indent=2))

## 8. Evaluate: Fine-Tuned vs Base Model

In [ ]:
print("Evaluating fine-tuned model...")
trained_model = SentenceTransformer(OUTPUT_DIR)
ft_results = evaluator(trained_model)

# Save fine-tuned results
with open(os.path.join(OUTPUT_DIR, "eval_results.json"), "w") as f:
    json.dump({k: float(v) if isinstance(v, (int, float)) else str(v) for k, v in ft_results.items()}, f, indent=2)

del trained_model
gc.collect()
torch.cuda.empty_cache()

# Evaluate base model for comparison
print("\nEvaluating base model for comparison...")
base_model = SentenceTransformer(BASE_MODEL)
base_model.max_seq_length = MAX_SEQ_LENGTH
base_results = evaluator(base_model)

with open(os.path.join(OUTPUT_DIR, "base_eval_results.json"), "w") as f:
    json.dump({k: float(v) if isinstance(v, (int, float)) else str(v) for k, v in base_results.items()}, f, indent=2)

del base_model
gc.collect()
torch.cuda.empty_cache()

# Compare key metrics
print(f"\n{'='*70}")
print("COMPARISON: BASE vs FINE-TUNED")
print(f"{'='*70}")
print(f"{'Metric':<25} {'Base':<12} {'Fine-tuned':<12} {'Improvement':<15}")
print("-" * 70)

key_metrics = ["mrr@10", "accuracy@1", "accuracy@5", "accuracy@10", "recall@10", "ndcg@10"]

for metric_suffix in key_metrics:
    ft_key = [k for k in ft_results if metric_suffix in k.lower()]
    base_key = [k for k in base_results if metric_suffix in k.lower()]

    if ft_key and base_key:
        ft_val = float(ft_results[ft_key[0]])
        base_val = float(base_results[base_key[0]])
        improvement = ft_val - base_val
        pct = (improvement / base_val * 100) if base_val > 0 else 0
        print(f"  {metric_suffix:<23} {base_val:<12.4f} {ft_val:<12.4f} +{improvement:.4f} ({pct:+.1f}%)")

print(f"{'='*70}")

# Save comparison
comparison = {}
for metric_suffix in key_metrics:
    ft_key = [k for k in ft_results if metric_suffix in k.lower()]
    base_key = [k for k in base_results if metric_suffix in k.lower()]
    if ft_key and base_key:
        comparison[metric_suffix] = {
            "base": float(base_results[base_key[0]]),
            "finetuned": float(ft_results[ft_key[0]]),
        }

with open(os.path.join(OUTPUT_DIR, "comparison.json"), "w") as f:
    json.dump(comparison, f, indent=2)

print(f"\nAll results saved to: {OUTPUT_DIR}")